In [1]:
import pandas as pd

results = pd.read_csv(
    "../data/raw/results.csv"
)

results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [2]:
results.info()

<class 'pandas.DataFrame'>
RangeIndex: 49472 entries, 0 to 49471
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49472 non-null  str    
 1   home_team   49472 non-null  str    
 2   away_team   49472 non-null  str    
 3   home_score  49400 non-null  float64
 4   away_score  49400 non-null  float64
 5   tournament  49472 non-null  str    
 6   city        49472 non-null  str    
 7   country     49472 non-null  str    
 8   neutral     49472 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 3.1 MB


In [3]:
results.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral'],
      dtype='str')

In [4]:
results.describe()

,home_score,away_score
count,49400.000000,49400.000000
mean,1.757247,1.181862
std,1.774311,1.402067
min,0.000000,0.000000
25%,1.000000,0.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,31.000000,21.000000


In [5]:
results.isnull().sum()

date           0
home_team      0
away_team      0
home_score    72
away_score    72
tournament     0
city           0
country        0
neutral        0
dtype: int64

In [6]:
results_clean = results.dropna(
    subset=["home_score","away_score"]
)

results_clean.shape

(49400, 9)

In [7]:
results_clean.isnull().sum()

date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64

In [8]:
results_clean["date"] = pd.to_datetime(
    results_clean["date"]
)

results_clean.dtypes

date          datetime64[us]
home_team                str
away_team                str
home_score           float64
away_score           float64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object

In [9]:
teams = pd.concat(
    [
        results_clean["home_team"],
        results_clean["away_team"]
    ]
).unique()

len(teams)

336

In [10]:
teams[:20]

<StringArray>
[        'Scotland',          'England',            'Wales',
 'Northern Ireland',    'United States',          'Uruguay',
          'Austria',          'Hungary',        'Argentina',
          'Belgium',           'France',         'Guernsey',
           'Jersey',      'Netherlands',           'Guyana',
   'Czechoslovakia',         'Alderney',      'Switzerland',
           'Sweden',          'Germany']
Length: 20, dtype: str

In [12]:
def match_result(row):
    
    if row["home_score"] > row["away_score"]:
        return "Home Win"
    
    elif row["home_score"] < row["away_score"]:
        return "Away Win"
    
    else:
        return "Draw"

In [13]:
results_clean["result"] = results_clean.apply(
    match_result,
    axis=1
)

results_clean[
    [
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "result"
    ]
].head()

,home_team,away_team,home_score,away_score,result
0,Scotland,England,0.0,0.0,Draw
1,England,Scotland,4.0,2.0,Home Win
2,Scotland,England,2.0,1.0,Home Win
3,England,Scotland,2.0,2.0,Draw
4,Scotland,England,3.0,0.0,Home Win


In [14]:
home_matches = (
    results_clean
    .groupby("home_team")
    .size()
)

away_matches = (
    results_clean
    .groupby("away_team")
    .size()
)

matches_played = (
    home_matches
    .add(
        away_matches,
        fill_value=0
    )
)

In [15]:
team_stats = pd.DataFrame({
    
    "matches_played":
    matches_played
    
})

team_stats.head()

,matches_played
home_team,
Abkhazia,33.0
Afghanistan,146.0
Albania,398.0
Alderney,135.0
Algeria,616.0


In [16]:
home_wins = (
    results_clean[
        results_clean["result"]
        == "Home Win"
    ]
    .groupby("home_team")
    .size()
)

away_wins = (
    results_clean[
        results_clean["result"]
        == "Away Win"
    ]
    .groupby("away_team")
    .size()
)

wins = home_wins.add(
    away_wins,
    fill_value=0
)

In [17]:
team_stats["wins"] = wins

team_stats.fillna(
    0,
    inplace=True
)

team_stats.head()

,matches_played,wins
home_team,,
Abkhazia,33.0,15.0
Afghanistan,146.0,36.0
Albania,398.0,110.0
Alderney,135.0,5.0
Algeria,616.0,287.0


In [18]:
team_stats.sort_values(
    "wins",
    ascending=False
).head(20)

,matches_played,wins
home_team,,
Brazil,1059.0,672.0
England,1089.0,624.0
Germany,1031.0,599.0
Argentina,1069.0,592.0
Sweden,1101.0,541.0
South Korea,1007.0,538.0
Mexico,1003.0,514.0
France,935.0,477.0
Italy,893.0,477.0


In [19]:
team_stats["win_rate"] = (
    team_stats["wins"]
    / team_stats["matches_played"]
)

team_stats.head()

,matches_played,wins,win_rate
home_team,,,
Abkhazia,33.0,15.0,0.454545
Afghanistan,146.0,36.0,0.246575
Albania,398.0,110.0,0.276382
Alderney,135.0,5.0,0.037037
Algeria,616.0,287.0,0.465909


In [20]:
team_stats.sort_values(
    "win_rate",
    ascending=False
).head(20)

,matches_played,wins,win_rate
home_team,,,
Surrey,1.0,1.0,1.000000
Elba Island,1.0,1.0,1.000000
Asturias,1.0,1.0,1.000000
Maule Sur,2.0,2.0,1.000000
Kurdistan,5.0,4.0,0.800000
Canary Islands,4.0,3.0,0.750000
Yorkshire,7.0,5.0,0.714286
County of Nice,9.0,6.0,0.666667
Franconia,3.0,2.0,0.666667


In [21]:
team_stats_filtered = (
    team_stats[
        team_stats["matches_played"] >= 100
    ]
)

team_stats_filtered.sort_values(
    "win_rate",
    ascending=False
).head(20)

,matches_played,wins,win_rate
home_team,,,
Jersey,235.0,153.0,0.651064
Brazil,1059.0,672.0,0.634561
Guernsey,240.0,145.0,0.604167
Spain,783.0,461.0,0.588761
Germany,1031.0,599.0,0.580989
England,1089.0,624.0,0.573003
Iran,612.0,349.0,0.570261
Argentina,1069.0,592.0,0.553789
Tahiti,242.0,131.0,0.541322


In [22]:
home_goals = (
    results_clean
    .groupby("home_team")
    ["home_score"]
    .sum()
)

away_goals = (
    results_clean
    .groupby("away_team")
    ["away_score"]
    .sum()
)

goals_for = (
    home_goals
    .add(
        away_goals,
        fill_value=0
    )
)

team_stats["goals_for"] = goals_for

In [23]:
home_goals_against = (
    results_clean
    .groupby("home_team")
    ["away_score"]
    .sum()
)

away_goals_against = (
    results_clean
    .groupby("away_team")
    ["home_score"]
    .sum()
)

goals_against = (
    home_goals_against
    .add(
        away_goals_against,
        fill_value=0
    )
)

team_stats["goals_against"] = goals_against

In [24]:
team_stats["goal_difference"] = (
    team_stats["goals_for"]
    - team_stats["goals_against"]
)

team_stats.head()

,matches_played,wins,win_rate,goals_for,goals_against,goal_difference
home_team,,,,,,
Abkhazia,33.0,15.0,0.454545,54.0,26.0,28.0
Afghanistan,146.0,36.0,0.246575,142.0,291.0,-149.0
Albania,398.0,110.0,0.276382,377.0,589.0,-212.0
Alderney,135.0,5.0,0.037037,73.0,620.0,-547.0
Algeria,616.0,287.0,0.465909,949.0,613.0,336.0


In [25]:
team_stats.sort_values(
    "goal_difference",
    ascending=False
).head(20)

,matches_played,wins,win_rate,goals_for,goals_against,goal_difference
home_team,,,,,,
Brazil,1059.0,672.0,0.634561,2305.0,957.0,1348.0
England,1089.0,624.0,0.573003,2378.0,1041.0,1337.0
Germany,1031.0,599.0,0.580989,2320.0,1198.0,1122.0
Argentina,1069.0,592.0,0.553789,2028.0,1075.0,953.0
Spain,783.0,461.0,0.588761,1602.0,703.0,899.0
South Korea,1007.0,538.0,0.534260,1792.0,914.0,878.0
Netherlands,879.0,453.0,0.515358,1841.0,1076.0,765.0
Sweden,1101.0,541.0,0.491371,2171.0,1421.0,750.0
Mexico,1003.0,514.0,0.512463,1767.0,1054.0,713.0


In [26]:
team_stats_filtered.sort_values(
    "win_rate",
    ascending=False
).head(20)

,matches_played,wins,win_rate
home_team,,,
Jersey,235.0,153.0,0.651064
Brazil,1059.0,672.0,0.634561
Guernsey,240.0,145.0,0.604167
Spain,783.0,461.0,0.588761
Germany,1031.0,599.0,0.580989
England,1089.0,624.0,0.573003
Iran,612.0,349.0,0.570261
Argentina,1069.0,592.0,0.553789
Tahiti,242.0,131.0,0.541322


In [27]:
draws_home = (
    results_clean[
        results_clean["result"] == "Draw"
    ]
    .groupby("home_team")
    .size()
)

team_stats["draws"] = draws_home

team_stats["draws"] = (
    team_stats["draws"]
    .fillna(0)
)

In [28]:
team_stats["losses"] = (
    team_stats["matches_played"]
    - team_stats["wins"]
    - team_stats["draws"]
)

In [29]:
team_stats["goals_per_match"] = (
    team_stats["goals_for"]
    / team_stats["matches_played"]
)

In [30]:
team_stats["goals_against_per_match"] = (
    team_stats["goals_against"]
    / team_stats["matches_played"]
)

In [31]:
team_stats.head()

,matches_played,wins,win_rate,goals_for,goals_against,goal_difference,draws,losses,goals_per_match,goals_against_per_match
home_team,,,,,,,,,,
Abkhazia,33.0,15.0,0.454545,54.0,26.0,28.0,8.0,10.0,1.636364,0.787879
Afghanistan,146.0,36.0,0.246575,142.0,291.0,-149.0,12.0,98.0,0.972603,1.993151
Albania,398.0,110.0,0.276382,377.0,589.0,-212.0,48.0,240.0,0.947236,1.479899
Alderney,135.0,5.0,0.037037,73.0,620.0,-547.0,1.0,129.0,0.540741,4.592593
Algeria,616.0,287.0,0.465909,949.0,613.0,336.0,85.0,244.0,1.540584,0.995130


In [32]:
team_stats.to_excel(
    "../data/processed/team_stats_v1.xlsx"
)